# 21 · Validação temporal: cinco desenhos da literatura, para frente, para trás e em blocos

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Pedido do autor (18/11).** Não ficar só no teste de três em três anos: testar ano a ano, cinco para frente, cinco para trás, um período contra o outro, e dizer com base em que metodologia publicada cada teste é feito.

**De onde vem cada desenho (declarado antes dos resultados).**

| Desenho | O que faz | Base na literatura |
|---|---|---|
| D1 · origem móvel, janela expansiva | calibra com 2011 até o ano T e confere em T+1 até T+H, para H de 1 a 5 anos; repete para todo T com pelo menos 4 anos de calibração | Tashman (2000): avaliação fora da amostra com origem móvel, recalibração a cada origem e vários períodos de teste |
| D2 · origem móvel, janela fixa | o mesmo, calibrando só com os últimos 3, 5 ou 7 anos | Tashman (2000): janela fixa contra janela expansiva |
| D3 · espelho para trás | calibra com os anos posteriores e confere os anteriores | robustez; não é previsão. Cerqueira, Torgo e Mozetič (2020) recomendam, para séries não estacionárias, os métodos que respeitam a ordem do tempo; por isso D1 é o principal e D3 só confere a estabilidade |
| D4 · validação cruzada em blocos | divide 2011 a 2025 em blocos contíguos de 5, 3 ou 1 ano; cada bloco é conferido com o modelo calibrado em todos os outros | Bergmeir e Benítez (2012); Roberts et al. (2017) |
| D5 · um ano de fora, com lacuna | como o D4 de 1 ano, retirando também o ano anterior e o posterior ao conferido | Bergmeir e Benítez (2012), variante que afasta os vizinhos dependentes |

**Por que avaliar em vários períodos, e não em um só.** Na avaliação de métodos de identificação de pontos críticos, os critérios clássicos comparam dois períodos consecutivos (Cheng e Washington, 2008; Montella, 2010); Guo, Wu e Lord (2020) mostram que isso ignora a instabilidade temporal dos acidentes e propõem critérios sobre vários períodos. A leitura do resultado como taxa de predição, com pesos calibrados no passado e conferidos no futuro, é a de Chung e Fabbri (2003).

**Por que três anos como horizonte principal.** A orientação de triagem de rede da Federal Highway Administration usa a média dos três a cinco anos mais recentes de acidentes; três anos é o menor horizonte dessa faixa e o que deixa mais origens. Os horizontes de 1 a 5 anos são todos reportados.

**Regras.** Nota só com atributos da via (regressão logística do notebook 18, hiperparâmetros fixos); no D1 também o modelo com histórico de acidentes. Todas as avaliações são reportadas; nenhuma é escolhida como resultado principal depois de vista. Avaliações com menos de 20 quilômetros atingidos na calibração ou na conferência são descartadas.

**Gates.** 20.936 células; 2.681 descarrilamentos por via; soma da matriz ano por célula igual a 2.681; D1 com H = 3 e origens de 2015 a 2022 reproduz o notebook 18 (0,752 sem histórico e 0,791 com histórico, tolerância de 0,002).

In [ ]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from joblib import Parallel, delayed
SEED = 42
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv", low_memory=False)
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]; gate("descarrilamentos por via", len(EV), 2681, source="notebook 11")
N = len(C); idx = pd.Series(np.arange(N), index=C["cel_id"].values)
def matriz(ev):
    M_ = np.zeros((15, N))
    for k in range(15):
        s = ev[ev["ano"] == 2011 + k].groupby("cel_id").size(); M_[k, idx[s.index].values] = s.values
    return M_
CV_, CA_ = matriz(EV), matriz(E); gate("soma da matriz ano x célula", int(CV_.sum()), 2681, source="partição por ano")
ANOS = np.arange(2011, 2026)
def conta(M_, anos): return M_[np.isin(ANOS, anos)].sum(0)
NUM = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)] + ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)] + ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
for c in NUM: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
Xn = C[NUM].copy(); med = Xn.median().fillna(0.0)
for c in NUM: Xn[c + "_na"] = Xn[c].isna().astype(float); Xn[c] = Xn[c].fillna(med[c])
X0 = StandardScaler().fit_transform(pd.concat([Xn, pd.get_dummies(C[CAT], dtype=float)], axis=1).values.astype(float))   # padronização só usa atributos, nunca a resposta
FERC = pd.factorize(C["ferrovia"])[0]; LINC = pd.factorize(C["ferrovia"].astype(str) + "|" + C["linha"].astype(str))[0]; SEGC = pd.factorize(C["seg_id"].astype(str))[0]; KM = ((C["km_a"] + C["km_b"]) / 2).values; LN = C["len_km"].values
ri, ci = [], []
for l in range(LINC.max() + 1):
    i = np.where(LINC == l)[0]; k = KM[i]; M_ = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M_); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(N, N)); nviz = np.asarray(W.sum(1)).ravel()
def historico(anos):
    """Histórico de acidentes com os anos informados (todos anteriores ao período conferido), por ano de observação."""
    n = max(len(anos), 1); hv = conta(CV_, anos); ha = conta(CA_, anos); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    return np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEGC)["b"].transform("mean").values, d.groupby(LINC)["b"].transform("mean").values, d.groupby(FERC)["b"].transform("mean").values, d.groupby(FERC)["hv"].transform("mean").values / n])
dp = C["dmin_patio"].fillna(1e9).values; rg = C["rmin_geo"].fillna(1e6).values
B2 = 0.5 * np.select([dp <= 300, dp <= 600, dp <= 1200, dp <= 1800], [4, 3, 2, 1], 0) / 4 + 0.5 * np.select([rg < 500, rg < 1200, rg < 2300], [3, 2, 1], 0) / 3
def auc(y, s):
    n1 = y.sum(); n0 = len(y) - n1; return float((rankdata(s)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 >= 5 and n0 >= 5 else np.nan
def auc_dentro(s, y):
    a, w = [], []
    for f in range(FERC.max() + 1):
        i = FERC == f
        if 5 <= y[i].sum() < i.sum(): a.append(auc(y[i], s[i])); w.append(y[i].sum())
    return float(np.average(a, weights=w)) if a else np.nan
DES = np.random.default_rng(SEED).random(N)
def classe_muito_alta(s, c):
    """Fração dos descarrilamentos conferidos que cai no quintil superior da nota (classe muito alta, cinco classes por quantis da extensão)."""
    o = np.lexsort((DES, -s)); cum = np.cumsum(LN[o]) / LN.sum(); return float(c[o[cum <= 0.20]].sum() / c.sum())
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
def avalia(desenho, cal, conf, com_historico=False, rotulo=""):
    """cal e conf: listas de anos, disjuntas. Devolve uma linha por método."""
    cal, conf = sorted(cal), sorted(conf); assert not set(cal) & set(conf), "janelas sobrepostas"
    ycal = (conta(CV_, cal) > 0).astype(int); cc = conta(CV_, conf); y = (cc > 0).astype(int)
    if ycal.sum() < 20 or y.sum() < 20: return []
    sentido = "para frente" if max(cal) < min(conf) else ("para trás" if min(cal) > max(conf) else "blocos (passado e futuro)")
    base = dict(desenho=desenho, rotulo=rotulo, calibra=f"{min(cal)}-{max(cal)}" + ("" if len(cal) == max(cal) - min(cal) + 1 else " (com lacuna)"), confere=f"{min(conf)}-{max(conf)}", anos_calibracao=len(cal), horizonte=len(conf), sentido=sentido, conf_ini=min(conf), conf_fim=max(conf), km_calibracao=int(ycal.sum()), km_conferidos=int(y.sum()))
    S = {"Regra do EAR por faixas": B2, "Dummy: taxa da malha": pd.Series(ycal).groupby(FERC).transform("mean").values, "EAR calibrado (atributos da via)": lr(X0, ycal, X0)}
    if com_historico and sentido == "para frente":
        H = len(conf); cal_h = [a for a in cal if a <= max(cal) - H]
        if len(cal_h) >= 2:
            alvo = [a for a in cal if a > max(cal) - H]; Htr = historico(cal_h); Hte = historico(cal); st = StandardScaler().fit(Htr); ytr = (conta(CV_, alvo) > 0).astype(int)
            if ytr.sum() >= 20: S["EAR calibrado + histórico de acidentes"] = lr(np.column_stack([X0, st.transform(Htr)]), ytr, np.column_stack([X0, st.transform(Hte)]))
    return [dict(base, metodo=m, auc=auc(y, s), auc_dentro=auc_dentro(s, y), classe_muito_alta=classe_muito_alta(s, cc)) for m, s in S.items()], (desenho, rotulo, tuple(cal), tuple(conf), y, S)
J = []
# D1: origem móvel, janela expansiva, para frente (Tashman, 2000); horizontes de 1 a 5 anos; mínimo de 4 anos de calibração
for H in (1, 2, 3, 4, 5):
    for T in range(2014, 2026 - H): J.append(("D1 origem móvel, janela expansiva", list(range(2011, T + 1)), list(range(T + 1, T + H + 1)), True, f"H={H}"))
# D2: origem móvel, janela fixa de 3, 5 e 7 anos, para frente
for Wd in (3, 5, 7):
    for H in (1, 3, 5):
        for T in range(2010 + Wd, 2026 - H): J.append(("D2 origem móvel, janela fixa", list(range(T - Wd + 1, T + 1)), list(range(T + 1, T + H + 1)), False, f"W={Wd}, H={H}"))
# D3: espelho para trás do D1 (robustez; não é previsão)
for H in (1, 3, 5):
    for T in range(2010 + H, 2023): J.append(("D3 espelho para trás", list(range(T + 1, 2026)), list(range(T - H + 1, T + 1)), False, f"H={H}"))
# D4: validação cruzada em blocos contíguos (Bergmeir e Benítez, 2012; Roberts et al., 2017): calibra em todos os outros blocos
for K, tam in ((3, 5), (5, 3), (15, 1)):
    for b in range(K):
        conf = list(range(2011 + b * tam, 2011 + (b + 1) * tam)); J.append(("D4 validação cruzada em blocos", [a for a in ANOS if a not in conf], conf, False, f"K={K} blocos de {tam} ano(s)"))
# D5: deixar um ano de fora com lacuna de um ano de cada lado (variante h-bloco: retira os vizinhos imediatos do ano conferido)
for a0 in ANOS: J.append(("D5 um ano de fora, com lacuna", [a for a in ANOS if abs(a - a0) > 1], [int(a0)], False, "lacuna de 1 ano"))
print("avaliações declaradas:", len(J)); gate_between("número de avaliações declaradas", float(len(J)), 150, 260, source="desenho")

log_ia("21.1 desenhos declarados", "Claude", "FOCUS: cinco desenhos de validação temporal com referência; só atributos da via, mais histórico no D1", "lista J", "3 gates de contagem; gate do número de avaliações")

In [ ]:
import time; t0 = time.time()
OUT = Parallel(n_jobs=2, batch_size=2)(delayed(avalia)(*j) for j in J); OUT = [o for o in OUT if o]
R = pd.DataFrame([l for o in OUT for l in o[0]]); GUARDA = [o[1] for o in OUT]; print("minutos:", round((time.time() - t0) / 60, 1), "| avaliações válidas:", len(OUT), "| linhas:", len(R))
R.to_csv(f"{TAB}/21_validacao_temporal_por_avaliacao.csv", index=False)
MET = ["Regra do EAR por faixas", "Dummy: taxa da malha", "EAR calibrado (atributos da via)", "EAR calibrado + histórico de acidentes"]
g = R.groupby(["desenho", "rotulo", "metodo"], sort=False)[["auc", "auc_dentro", "classe_muito_alta"]].agg(["mean", "min", "max"]); g[("n", "")] = R.groupby(["desenho", "rotulo", "metodo"], sort=False).size(); g = g.round(3); g.to_csv(f"{TAB}/21_validacao_temporal_resumo.csv")
res = R.pivot_table(index=["desenho", "rotulo"], columns="metodo", values="auc", aggfunc="mean", sort=False)[[m for m in MET if m in R.metodo.unique()]].round(3); res["n"] = R[R.metodo == MET[0]].groupby(["desenho", "rotulo"], sort=False).size(); display(res)
print("== AUC dentro da malha =="); display(R.pivot_table(index=["desenho", "rotulo"], columns="metodo", values="auc_dentro", aggfunc="mean", sort=False)[[m for m in MET if m in R.metodo.unique()]].round(3))
print("== Por desenho (todas as configurações) =="); d = R.groupby(["desenho", "metodo"], sort=False)["auc"].agg(["mean", "min", "max", "count"]).round(3); display(d); d.to_csv(f"{TAB}/21_validacao_temporal_por_desenho.csv")
# reprodução do notebook 18: D1, H=3, origens de 2015 a 2022
p = R[(R.desenho.str.startswith("D1")) & (R.rotulo == "H=3") & (R.conf_ini >= 2016) & (R.metodo == "EAR calibrado (atributos da via)")]["auc"].mean(); gate("D1 com H=3 e origens de 2015 a 2022 reproduz o notebook 18", round(float(p), 3), 0.752, tol=0.002, source="notebook 18")
p = R[(R.desenho.str.startswith("D1")) & (R.rotulo == "H=3") & (R.conf_ini >= 2016) & (R.metodo == "EAR calibrado + histórico de acidentes")]["auc"].mean(); gate("idem, com histórico", round(float(p), 3), 0.791, tol=0.002, source="notebook 18")
pv = R.pivot_table(index=["desenho", "rotulo", "calibra", "confere"], columns="metodo", values="auc"); dif = pv["EAR calibrado (atributos da via)"] - pv["Regra do EAR por faixas"]
print("EAR calibrado acima da regra em", int((dif > 0).sum()), "de", int(dif.notna().sum()), "avaliações; diferença média", round(dif.mean(), 3), "| mínima", round(dif.min(), 3), "| máxima", round(dif.max(), 3))
# intervalo de 95% da média por desenho, por reamostragem de linhas inteiras (notas fixas; 300 reamostragens)
rng = np.random.default_rng(SEED); cel_lin = [np.where(LINC == l)[0] for l in range(LINC.max() + 1)]; amostras = [np.concatenate([cel_lin[l] for l in rng.integers(0, len(cel_lin), len(cel_lin))]) for _ in range(300)]; L = []
for chave in dict.fromkeys((gq[0], gq[1]) for gq in GUARDA):
    itens = [gq for gq in GUARDA if (gq[0], gq[1]) == chave]
    for m in MET:
        im = [gq for gq in itens if m in gq[5]]
        if not im: continue
        bs = [np.nanmean([auc(gq[4][ii], gq[5][m][ii]) for gq in im]) for ii in amostras]; L.append(dict(desenho=chave[0], rotulo=chave[1], metodo=m, n=len(im), media=np.nanmean([auc(gq[4], gq[5][m]) for gq in im]), ic95_inf=np.nanpercentile(bs, 2.5), ic95_sup=np.nanpercentile(bs, 97.5)))
IC = pd.DataFrame(L).round(3); IC.to_csv(f"{TAB}/21_validacao_temporal_ic95.csv", index=False); display(IC[IC.metodo.str.startswith("EAR calibrado")])
log_ia("21.2 bateria executada", "Claude", "FOCUS: cinco desenhos declarados; todos reportados; IC por reamostragem de linhas", "21_validacao_temporal_*.csv", "2 gates de reprodução do notebook 18")


In [ ]:
# Figura 1 do artigo (rótulos em inglês, porque o artigo é em inglês)
import os, matplotlib, matplotlib.font_manager as fm                     # estilo da dissertação: Latin Modern Sans, tinta, cinzas, verde-petróleo e argila
for f_ in ("lmsans10-regular.otf", "lmsans10-bold.otf", "lmsans10-oblique.otf"):
    for d_ in (f"{BASE}/data/fonts", "/usr/share/texmf/fonts/opentype/public/lm"):
        if os.path.exists(f"{d_}/{f_}"): fm.fontManager.addfont(f"{d_}/{f_}"); break
FONTE = "Latin Modern Sans" if any(f.name == "Latin Modern Sans" for f in fm.fontManager.ttflist) else "DejaVu Sans"
matplotlib.rcParams.update({"font.family": FONTE, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#7A8089", "axes.linewidth": 0.6, "xtick.color": "#16191C", "ytick.color": "#16191C"})
import matplotlib.pyplot as plt
INK, MUTE, GRID, TEAL, CLAY, GOLD = "#16191C", "#7A8089", "#E6E8EB", "#1E6E6E", "#BD4B2C", "#8C6D1F"
COR = {"Regra do EAR por faixas": INK, "Dummy: taxa da malha": GOLD, "EAR calibrado (atributos da via)": TEAL, "EAR calibrado + histórico de acidentes": CLAY}
EN = {"Regra do EAR por faixas": "Expert rule (FICO bands)", "Dummy: taxa da malha": "Network-rate dummy", "EAR calibrado (atributos da via)": "Logistic score (track attributes)", "EAR calibrado + histórico de acidentes": "Logistic score + accident history"}
LS = {"Regra do EAR por faixas": (0, (4, 2)), "Dummy: taxa da malha": (0, (1, 1.5)), "EAR calibrado (atributos da via)": "-", "EAR calibrado + histórico de acidentes": "-"}
fig, ax = plt.subplots(1, 2, figsize=(8.4, 5.0), gridspec_kw={"width_ratios": [1.0, 1.12]})
a = ax[0]; d = R[(R.desenho.str.startswith("D1")) & (R.rotulo == "H=1")]
for m in MET:
    g = d[d.metodo == m].sort_values("conf_ini"); a.plot(g.conf_ini, g.auc, linestyle=LS[m], color=COR[m], lw=2, marker="o", ms=4.5, mec="white", mew=0.8, label=EN[m]); a.annotate(f"{g.auc.iloc[-1]:.2f}", (g.conf_ini.iloc[-1], g.auc.iloc[-1]), xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color=INK)
a.axhline(0.5, color=MUTE, lw=0.8); a.text(2025.9, 0.508, "chance", fontsize=8.5, color=MUTE, ha="right")
a.set_ylim(0.33, 0.90); a.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9]); a.set_xlim(2014.6, 2026.2); a.set_xticks(range(2015, 2026, 2)); a.grid(axis="y", color=GRID, lw=0.8); a.set_axisbelow(True)
a.set_xlabel("Checked year (calibrated on all earlier years)", fontsize=10, color=INK); a.set_ylabel("AUC, all 20,936 cells", fontsize=10, color=INK); a.set_title("(a) Rolling origin, one year ahead", loc="left", fontsize=10, color=INK, fontweight="bold"); a.legend(frameon=False, fontsize=8.4, loc="lower left", ncol=1, handlelength=2.4, borderaxespad=0.2, labelspacing=0.3)
b = ax[1]; ordem = [("D1 origem móvel, janela expansiva", f"H={h}", f"Expanding, {h} yr ahead") for h in (1, 2, 3, 4, 5)] + [("D2 origem móvel, janela fixa", f"W={w}, H={h}", f"Fixed {w}-yr, {h} yr ahead") for w in (3, 5, 7) for h in (1, 3, 5)] + [("D3 espelho para trás", f"H={h}", f"Backward, {h} yr") for h in (1, 3, 5)] + [("D4 validação cruzada em blocos", "K=3 blocos de 5 ano(s)", "Blocked CV, 3 × 5 yr"), ("D4 validação cruzada em blocos", "K=5 blocos de 3 ano(s)", "Blocked CV, 5 × 3 yr"), ("D4 validação cruzada em blocos", "K=15 blocos de 1 ano(s)", "Blocked CV, 15 × 1 yr"), ("D5 um ano de fora, com lacuna", "lacuna de 1 ano", "Leave-one-year-out")]
yy = np.arange(len(ordem))[::-1]
for y_, (de, ro, lab) in zip(yy, ordem):
    for m in MET:
        v = R[(R.desenho == de) & (R.rotulo == ro) & (R.metodo == m)]["auc"]
        if len(v) == 0 or m.startswith("Dummy"): continue
        dy = 0.22 if m.endswith("histórico de acidentes") else (-0.22 if (m.startswith("EAR calibrado (") and de.startswith("D1")) else 0.0)
        b.plot([v.min(), v.max()], [y_ + dy, y_ + dy], color=COR[m], lw=1.6, alpha=0.55, solid_capstyle="round"); b.plot(v.mean(), y_ + dy, "o" if not m.startswith("Regra") else "D", color=COR[m], ms=6 if not m.startswith("Regra") else 4.5, mec="white", mew=0.8)
b.set_yticks(yy); b.set_yticklabels([o[2] for o in ordem], fontsize=8.6, color=INK); b.set_xlim(0.5, 0.9); b.grid(axis="x", color=GRID, lw=0.8); b.set_axisbelow(True); b.tick_params(axis="y", length=0)
for sep in (15.5, 6.5, 3.5): b.axhline(sep, color=GRID, lw=0.8)
b.set_xlabel("AUC: mean (marker) and range (line)", fontsize=10, color=INK); b.set_title(f"(b) All {len(OUT)} evaluations, by design", loc="left", fontsize=10, color=INK, fontweight="bold")
fig.tight_layout(w_pad=1.2); fig.savefig(f"{FIG}/21_validacao_temporal.png", dpi=300, facecolor="white"); plt.show()
exportar_registros("21")


---
## Referências (registro conferido na Scopus em 18/11/2026, pelo acesso do IME; resumo lido)

Bergmeir, C., & Benítez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences, 191*, 192–213. https://doi.org/10.1016/j.ins.2011.12.028

Cerqueira, V., Torgo, L., & Mozetič, I. (2020). Evaluating time series forecasting models: An empirical study on performance estimation methods. *Machine Learning, 109*(11), 1997–2028. https://doi.org/10.1007/s10994-020-05910-7

Cheng, W., & Washington, S. (2008). New criteria for evaluating methods of identifying hot spots. *Transportation Research Record, 2083*, 76–85. https://doi.org/10.3141/2083-09

Chung, C.-J. F., & Fabbri, A. G. (2003). Validation of spatial prediction models for landslide hazard mapping. *Natural Hazards, 30*(3), 451–472. https://doi.org/10.1023/B:NHAZ.0000007172.62651.2b

Guo, X., Wu, L., & Lord, D. (2020). Generalized criteria for evaluating hotspot identification methods. *Accident Analysis & Prevention, 145*, 105684. https://doi.org/10.1016/j.aap.2020.105684

Montella, A. (2010). A comparative analysis of hotspot identification methods. *Accident Analysis & Prevention, 42*(2), 571–581. https://doi.org/10.1016/j.aap.2009.09.025

Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S., Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., & Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0

Federal Highway Administration. (s.d.). *Step 2. Conduct network screening*. Improving Safety on Rural Local and Tribal Roads: Safety Toolkit. https://highways.dot.gov/safety/local-rural/improving-safety-rural-local-and-tribal-roads-safety-toolkit/step-2-conduct (lida em 18/11/2026)

## Limites

Os atributos da via são os da Declaração de Rede de 2025 em todas as avaliações. As avaliações se sobrepõem no tempo e não são independentes: a média resume cada desenho e o intervalo por reamostragem de linhas trata a dependência espacial, não a temporal. No D2, as janelas mais longas só existem para origens mais recentes, o que confunde o efeito do tamanho da janela com o do período conferido. No D4 e no D5 a calibração usa anos posteriores ao conferido; servem para medir estabilidade, não para simular o uso.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.